In [ ]:
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
import joblib
from utils.data_loader import load_as_maps
from models.unet import build_unet

In [ ]:
# Specify datasets that should be used for training
datasets=["exp1","exp5","exp6","exp7"]

# Specify index of target variable
# 0 for fco2
# 1 for fco2_pre 
# 2 for co2flux 
# 3 for co2flux_pre
target_index = 3 

# Specify model parameters
lr = 0.0005
batch_size = 4
base_filters = 64
dropout_rate = 0.05
kernel_size= (3,3)
n_epochs = 100

In [ ]:
# load data
features, targets = load_as_maps(datasets=datasets, target_index=target_index)

# split data
X_train = features[:int(0.8 * len(features))]
Y_train = targets[:int(0.8 * len(targets))]
X_val = features[int(0.8 * len(features)):int(0.9 * len(features))]
Y_val = targets[int(0.8 * len(targets)):int(0.9 * len(targets))]
X_test = features[int(0.9 * len(features)):]
Y_test = targets[int(0.9 * len(targets)):]

# scale data
scaler = MinMaxScaler()
n_samples, h, w, n_features = X_train.shape
X_train_flat = X_train.reshape(-1,n_features)
X_train_scaled_flat = scaler.fit_transform(X_train_flat)
X_train = X_train_scaled_flat.reshape(n_samples, h, w, n_features)


n_samples, h, w, n_features = X_val.shape
X_val_flat = X_val.reshape(-1,n_features)
X_val_scaled_flat = scaler.transform(X_val_flat)
X_val = X_val_scaled_flat.reshape(n_samples, h, w, n_features)

n_samples, h, w, n_features = X_test.shape
X_test_flat = X_test.reshape(-1,n_features)
X_test_scaled_flat = scaler.transform(X_test_flat)
X_test = X_test_scaled_flat.reshape(n_samples, h, w, n_features)

# save scaler
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')
folder_path = "../../outputs/u-net/" + timestamp
os.makedirs(folder_path, exist_ok=True)

joblib.dump(scaler, folder_path + '/scaler.pkl')

In [ ]:
# build model
model = build_unet((167, 360, 13),base_filters,kernel_size,dropout_rate)
model.compile(optimizer=tf.keras.optimizers.Nadam(learning_rate=lr), loss='mse', metrics=['mae'])
model.summary()

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

H, W = 167, 360
mask_ch = 10

if Y_train.ndim == 3: Y_train = Y_train[..., None]
if Y_val.ndim   == 3: Y_val   = Y_val[..., None]

m_train = X_train[..., mask_ch]
m_val   = X_val[..., mask_ch]

history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val, m_val),
    batch_size=batch_size,  
    epochs=n_epochs,  
    sample_weight=m_train,
    callbacks=[lr_scheduler, early_stopping],
    shuffle=True
)

In [ ]:
model.save(folder_path + "/model.keras")

plt.plot(history.history['loss'], color = 'blue', label = 'training loss')
plt.plot(history.history['val_loss'], color = 'red', label = 'validation loss')
plt.xlabel('Epoch number')
plt.ylabel('Loss')
plt.title('Training and validation loss')
plt.grid(True)
plt.legend()

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')
path = folder_path + '/' + timestamp + '_training_validation_loss.png'
plt.savefig(path, format='png', dpi=300)
plt.show()  

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

test_pred = model.predict(X_test)
pred = test_pred.reshape(-1)
truth = Y_test.reshape(-1)

# computiung mse and mae for ocean points only
mask = X_test.reshape(-1,13)
mask = mask[:, 10] == 1
pred = pred[mask]
truth = truth[mask]

mse = mean_squared_error(pred, truth)
mae = mean_absolute_error(pred, truth)
print(f"Mean Squared Error: {mse:.3f}")
print(f"Mean Absolute Error: {mae:.3f}")

In [ ]:
from utils.model_analysis import complete_model_analysis_map

for dataset_id in datasets:
    complete_model_analysis_map(folder_path,dataset_id)